# store

> The `learning` shelf: concepts, attempts and sessions as vault documents.

This module keeps all learning data in the vishalakshi vault, on one shelf named `learning`.
Each concept, attempt and session is one vault document, so every tool that reads the vault can search it.
The other modules read and write the shelf only through the `Learning` class in this module.

In [ ]:
#| default_exp store

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import json, re, time, uuid, unicodedata
from pathlib import Path
from fastcore.utils import *
from vishalakshi import Vault

## The shelf

Search reads the text of a document. The `meta` JSON keeps the state.
`Vault.set_meta` changes the state and does not embed the text again.

| kind | source |
|---|---|
| `concept` | `learning://concept/<slug>` |
| `attempt` | `learning://attempt/<id>` |
| `session` | `learning://session/<id>` |

These constants name the shelf, give the start state of a concept, and list the fields that merge.

In [ ]:
#| export
SHELF = 'learning'
#: the learner state of a concept that has no attempts yet
STATE = dict(floor=0, ceiling=6, stability=0., due=None, last=None, streak=0, misses=0, seen=0, shaky=None, misconceptions=[])
#: content fields that a new source adds to, and does not replace
LISTS = ('prereqs', 'related', 'aliases', 'provenance', 'questions')
_SKIP = ('content_hash', 'pii_scanned')

`slugify` makes the key of a concept from its name. `_src` makes the source URI of a document.

In [ ]:
#| export
def slugify(s:str) -> str:
    "Make a lower-case ASCII slug from `s`; marks are dropped, so `saṃdhi` gives `samdhi`."
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '-', s.lower()).strip('-') or uuid.uuid4().hex[:8]

def _src(kind, slug): return f'learning://{kind}/{slug}'

`_union` merges two lists and keeps the order. `concept_text` makes the text that search reads.

In [ ]:
#| export
def _union(a, b):
    "Items of `a`, then the new items of `b`, in order, compared as JSON."
    seen, out = set(), []
    for x in (*(a or ()), *(b or ())):
        k = json.dumps(x, sort_keys=True, default=str)
        if k not in seen: seen.add(k); out.append(x)
    return out

def concept_text(c) -> str:
    "The searchable text of a concept: its name, statement, aliases and subject."
    al = ', '.join(c.get('aliases') or [])
    return '\n\n'.join(o for o in (c['name'], c.get('statement', ''), al and f'Also: {al}',
                                   c.get('subject') and f'Subject: {c["subject"]}') if o)

`Learning` opens one shelf of a vault. It also accepts a path, or `None` for the default vault.

In [ ]:
#| export
class Learning:
    "Concepts, attempts and sessions on one shelf of a vishalakshi vault."
    def __init__(self, vault=None, shelf:str=SHELF):
        if not isinstance(vault, Vault): vault = Vault(vault)
        self.vault, self.v = vault, vault.shelf(shelf)

    def _docs(self, kind, where='', args=()):
        w = 'kind=?' + (f' and ({where})' if where else '')
        return self.v.t.docs(where=w, where_args=[kind, *args])

    def _row(self, r):
        m = {k: v for k, v in json.loads(r['meta'] or '{}').items() if k not in _SKIP}
        return AttrDict({**STATE, **m, 'doc_id': r['id'], 'name': m.get('name', r['title'])})

    def __repr__(self): return f'Learning({self.v.path!r}: {len(self._docs("concept"))} concepts)'

## Read concepts

Each method returns concepts as `AttrDict`s with the learner state filled in.

In [ ]:
#| export
@patch
def concept(self:Learning, slug:str) -> AttrDict|None:
    "One concept by slug, with its learner state."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('concept', slug)]))
    return self._row(r) if r else None

@patch
def concepts(self:Learning, subject:str=None) -> L:
    "All concepts, or the concepts of one subject."
    rs = self._docs('concept', *(("json_extract(meta, '$.subject')=?", [subject]) if subject else ()))
    return L(rs).map(self._row)

@patch
def subjects(self:Learning) -> list:
    "The subjects that have concepts, with counts."
    return self.v.db.q(f"select json_extract(meta, '$.subject') as subject, count(*) as n from [{self.v.t.prefix}docs] "
                       "where kind='concept' group by 1 order by 2 desc")

## Write concepts

`put` adds a concept or adds to it. It keeps the learner state, merges the list fields and replaces the other fields.

In [ ]:
#| export
@patch
def put(self:Learning,
        name:str,              # the concept's name
        statement:str='',      # a true, self-contained statement of the idea
        subject:str='',        # e.g. 'sanskrit', 'jyotisha', 'ml'
        slug:str=None,         # defaults to `slugify(name)`
        prereqs=(),            # slugs this concept needs first
        related=(),            # slugs linked to it, in any direction
        aliases=(),            # other names
        provenance=(),         # where it came from: dicts such as `{'src': 'vedicreader', 'ref': ...}`
        questions=(),          # ready questions: `{'q':..., 'a':..., 'depth':...}`
        **meta) -> AttrDict:
    "Add a concept or add to one. Keep the learner state, merge the list fields and replace the other fields."
    slug = slug or slugify(name)
    old = self.concept(slug) or {}
    new = dict(name=name, statement=statement or old.get('statement', ''), subject=subject or old.get('subject', ''), slug=slug, **meta)
    for k, v in dict(prereqs=prereqs, related=related, aliases=aliases, provenance=provenance, questions=questions).items():
        new[k] = _union(old.get(k), v)
    new['prereqs'] = [p for p in new['prereqs'] if p != slug]
    m = {**STATE, **{k: v for k, v in old.items() if k != 'doc_id'}, **new}
    r = self.v.add(concept_text(m), title=name, source=_src('concept', slug), kind='concept', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return self.concept(slug)

`update` and `link` change one concept. `drop` removes it.

In [ ]:
#| export
@patch
def update(self:Learning, slug:str, **kv) -> AttrDict:
    "Change the state of a concept and do not embed its text again."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    self.v.set_meta(c.doc_id, **kv)
    return self.concept(slug)

@patch
def link(self:Learning, slug:str, prereq:str):
    "Record that `prereq` comes before `slug`."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    if prereq != slug and prereq not in c.prereqs: self.update(slug, prereqs=[*c.prereqs, prereq])

@patch
def drop(self:Learning, slug:str):
    "Remove a concept. Its attempts stay as a record of what happened."
    if (c := self.concept(slug)): self.v.forget(c.doc_id)

## Attempts

Each answer is one attempt document. Search can find what the learner said, and the tutor can avoid old questions.

In [ ]:
#| export
def attempt_text(a) -> str:
    "The searchable text of an attempt: the question, the answer, the think-aloud and the feedback."
    parts = [f"Question: {a['question']}", a.get('answer') and f"Answer: {a['answer']}",
             a.get('thinking') and f"Thinking: {a['thinking']}", a.get('feedback') and f"Feedback: {a['feedback']}"]
    return '\n\n'.join(p for p in parts if p)

`record` files one attempt.

In [ ]:
#| export
@patch
def record(self:Learning, concept:str, question:str, answer:str='', thinking:str='', correct:bool=False,
           confidence:int=50, depth:int=1, kind:str='explain', hints:int=0, dont_know:bool=False, grade:int=None,
           feedback:str='', first_error:str='', misconception:str='', session:str='', **meta) -> AttrDict:
    "File one attempt as a vault document, so the learner can search what they said."
    a = AttrDict(id=uuid.uuid4().hex[:12], concept=concept, question=question, answer=answer, thinking=thinking,
                 correct=correct, confidence=confidence, depth=depth, kind=kind, hints=hints, dont_know=dont_know,
                 grade=grade, feedback=feedback, first_error=first_error, misconception=misconception,
                 session=session, at=time.time(), **meta)
    c = self.concept(concept)
    self.v.add(attempt_text(a), title=f"{c.name if c else concept}: {question[:80]}", source=_src('attempt', a.id),
               kind='attempt', meta=dict(a))
    return a

`attempts` reads them back, with the newest first.

In [ ]:
#| export
@patch
def attempts(self:Learning, concept:str=None, session:str=None, limit:int=50) -> L:
    "Recent attempts, newest first, for one concept or one session."
    w, args = [], []
    if concept: w.append("json_extract(meta, '$.concept')=?"); args.append(concept)
    if session: w.append("json_extract(meta, '$.session')=?"); args.append(session)
    rs = L(self._docs('attempt', ' and '.join(w), args)).map(lambda r: AttrDict(json.loads(r['meta'])))
    return L(sorted(rs, key=lambda a: -a.at)[:limit]).map(lambda a: AttrDict({k: v for k, v in a.items() if k not in _SKIP}))

## Sessions

A session summary is one document. A new summary with the same id replaces the old one.

In [ ]:
#| export
@patch
def log_session(self:Learning, sid:str, summary:str, **meta):
    "File a session summary, and replace an earlier one with the same id."
    return self.v.add(summary, title=f'Session {sid}', source=_src('session', sid), kind='session', meta=dict(id=sid, **meta))

## Search and graph

`search` uses the hybrid search of the vault. `graph` gives the nodes and links for the graph view.

In [ ]:
#| export
@patch
def search(self:Learning, q:str, kind:str=None, limit:int=10) -> L:
    "Hybrid search over the shelf: concepts, attempts and sessions."
    return L(self.v.search(q, limit=limit, kind=kind))

@patch
def graph(self:Learning, subject:str=None) -> dict:
    "Nodes and links for a graph view. A link to a slug that has no concept is dropped."
    cs = self.concepts(subject)
    have = set(cs.attrgot('slug'))
    nodes = [dict(id=c.slug, name=c.name, subject=c.subject, floor=c.floor, ceiling=c.ceiling, due=c.due, seen=c.seen) for c in cs]
    links = [dict(source=p, target=c.slug, kind='prereq') for c in cs for p in c.prereqs if p in have]
    links += [dict(source=c.slug, target=r, kind='related') for c in cs for r in c.related if r in have and c.slug < r]
    return dict(nodes=nodes, links=links)

## A shelf for tests

Each test uses a vault in a temporary folder with the offline hashing encoder.
The tests never open the learner's real vault.

In [ ]:
#| export
def temp_learning(**kw) -> Learning:
    "A `Learning` on a new vault in a temporary folder, with the offline hashing encoder."
    import tempfile
    return Learning(Vault(str(Path(tempfile.mkdtemp())/'vault.db'), offline=True), **kw)

## Tests

`put` makes a concept with the start state and the slug from its name.

In [ ]:
lr = temp_learning()
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank) for each item.', 'ml', prereqs=['ranking'], aliases=['RRF'])
test_eq(c.slug, 'reciprocal-rank-fusion')
test_eq(c.floor, 0)
test_eq(c.prereqs, ['ranking'])

`update` changes the state.

In [ ]:
c = lr.update(c.slug, floor=3, stability=2.)
test_eq((c.floor, c.stability), (3, 2.))

A second `put` keeps the learner state and merges the lists, also when the text changes.

In [ ]:
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists: score is the sum of 1/(k + rank).', related=['bm25'], aliases=['RRF', 'rank fusion'])
test_eq(c.floor, 3)
test_eq(c.subject, 'ml')
test_eq(c.aliases, ['RRF', 'rank fusion'])
test_eq(c.related, ['bm25'])
test_eq(len(lr.concepts()), 1)

`concepts` and `subjects` filter by subject. `graph` keeps only the links between known concepts.

In [ ]:
lr.put('Ranking', 'Order items by a score.', 'ml')
lr.put('Sandhi', 'Sounds change where words meet.', 'sanskrit')
test_eq(len(lr.concepts('ml')), 2)
test_eq({r['subject'] for r in lr.subjects()}, {'ml', 'sanskrit'})
g = lr.graph()
test_eq(g['links'], [dict(source='ranking', target='reciprocal-rank-fusion', kind='prereq')])

`record` files an attempt. `attempts` and `search` find it again.

In [ ]:
a = lr.record('reciprocal-rank-fusion', 'Why does RRF ignore raw scores?', answer='Because scores from different systems are not comparable.',
              thinking='bm25 and cosine have different scales', correct=True, confidence=80, depth=3)
test_eq(lr.attempts('reciprocal-rank-fusion')[0].id, a.id)
test_eq(lr.attempts('ranking'), [])
test(lr.search('scales comparable', kind='attempt'), [], lambda a,b: len(a) > 0)

`drop` removes a concept.

In [ ]:
lr.drop('ranking')
test_is(lr.concept('ranking'), None)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()